### Import

In [10]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

In [11]:
CHARADES_ROOT = r"D:\Datasets\Datasets\Charades"
RGB_ROOT = os.path.join(CHARADES_ROOT, "RGB")
FLOW_ROOT = os.path.join(CHARADES_ROOT, "Flow")
ANNOTATION_CSV = r"D:\Datasets\Datasets\Charades\Label\Charadesh_Annotation.csv"

OUTPUT_DIR = "../Features/Charades/Charades_features.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FEATURE_DIM = 512
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [13]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [18]:
unique_labels = sorted(annotations["ActionLabel"].unique())

label_map = {old: new for new, old in enumerate(unique_labels)}

print("Total classes:", len(label_map))

Total classes: 37


In [12]:
annotations = pd.read_csv(ANNOTATION_CSV)

# Rename for consistency
annotations.columns = ["video_id", "StartFrame", "EndFrame", "ActionLabel", "ActionName"]

# Remove invalid rows
annotations = annotations[annotations["StartFrame"] <= annotations["EndFrame"]]

print("Total valid annotations:", len(annotations))

Total valid annotations: 62


In [19]:
annotations["MappedLabel"] = annotations["ActionLabel"].map(label_map)

In [20]:
resnet = models.resnet50(weights=True)
modules = list(resnet.children())[:-1]
resnet = nn.Sequential(*modules)

projection = nn.Linear(2048, FEATURE_DIM)

resnet = resnet.to(DEVICE).eval()
projection = projection.to(DEVICE).eval()

C:\Users\PAWANESH\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [21]:
@torch.no_grad()
def extract_feature(image_path):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(DEVICE)

    feature = resnet(image)
    feature = feature.view(1, -1)
    feature = projection(feature)

    return feature.squeeze(0).cpu().numpy()

In [22]:
class AdaptiveFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, rgb, flow):
        x = torch.cat([rgb, flow], dim=1)
        alpha = self.sigmoid(self.fc(x))
        return alpha * rgb + (1 - alpha) * flow

fusion_model = AdaptiveFusion(FEATURE_DIM).to(DEVICE).eval()

In [23]:
video_ids = annotations["video_id"].unique()

for vid in video_ids:
    print(f"\nProcessing video: {vid}")
    rgb_path = os.path.join(RGB_ROOT, vid, "rgb")
    flow_path = os.path.join(FLOW_ROOT, vid, "flow")
    rgb_frames = sorted(glob.glob(os.path.join(rgb_path, "*.jpg")))
    flow_frames = sorted(glob.glob(os.path.join(flow_path, "*.jpg")))
    if len(rgb_frames) == 0 or len(flow_frames) == 0:
        print("Skipping (missing data)")
        continue
    if len(rgb_frames) != len(flow_frames):
        print("Skipping (frame mismatch)")
        continue
    num_frames = len(rgb_frames)

    # -------------------------
    # Label creation (PER VIDEO)
    # -------------------------
    video_ann = annotations[annotations["video_id"] == vid]
    num_classes = annotations["ActionLabel"].nunique()
    labels = np.zeros((num_frames, num_classes), dtype=np.float32)
    for _, row in video_ann.iterrows():
        s = int(row["StartFrame"])
        e = int(row["EndFrame"])
        cls = int(row["MappedLabel"])
        if s >= num_frames:
            continue
        e = min(e, num_frames - 1)
        labels[s:e+1, cls] = 1.0

    # -------------------------
    # Feature extraction
    # -------------------------
    rgb_features = np.zeros((num_frames, FEATURE_DIM), dtype=np.float32)
    flow_features = np.zeros((num_frames, FEATURE_DIM), dtype=np.float32)

    for i in tqdm(range(num_frames), desc=f"{vid}"):
        rgb_features[i] = extract_feature(rgb_frames[i])
        flow_features[i] = extract_feature(flow_frames[i])

    # -------------------------
    # Fusion
    # -------------------------
    rgb_tensor = torch.from_numpy(rgb_features).to(DEVICE)
    flow_tensor = torch.from_numpy(flow_features).to(DEVICE)

    with torch.no_grad():
        fused = fusion_model(rgb_tensor, flow_tensor)

    fused_features = fused.cpu().numpy()

    # -------------------------
    # Save
    # -------------------------
    feat_df = pd.DataFrame(fused_features)
    feat_df.insert(0, "frame_id", np.arange(num_frames))

    label_df = pd.DataFrame(labels)
    label_df.insert(0, "frame_id", np.arange(num_frames))

    feat_df.to_csv(os.path.join(OUTPUT_DIR, f"{vid}_features.csv"), index=False)
    label_df.to_csv(os.path.join(OUTPUT_DIR, f"{vid}_labels.csv"), index=False)

    print(f"Saved {vid}")


Processing video: 00HFP


00HFP: 100%|█████████████████████████████████████████████████████████████████████████| 746/746 [00:33<00:00, 22.07it/s]


Saved 00HFP

Processing video: 00MFE


00MFE: 100%|█████████████████████████████████████████████████████████████████████████| 493/493 [00:23<00:00, 20.67it/s]


Saved 00MFE

Processing video: 00N38


...

Saved 00X3U

Processing video: 00YZL


00YZL: 100%|█████████████████████████████████████████████████████████████████████████| 796/796 [00:37<00:00, 21.33it/s]


Saved 00YZL

Processing video: 00ZCA


00ZCA: 100%|███████████████████████████████████████████████████████████████████████| 2229/2229 [01:43<00:00, 21.44it/s]


Saved 00ZCA


In [9]:
for vid in video_ids:
    rgb_path = os.path.join(RGB_ROOT, vid, "rgb")
    flow_path = os.path.join(FLOW_ROOT, vid, "flow")

    rgb_frames = sorted(glob.glob(os.path.join(rgb_path, "*.jpg")))
    flow_frames = sorted(glob.glob(os.path.join(flow_path, "*.jpg")))

    print(f"{vid}: RGB={len(rgb_frames)}, FLOW={len(flow_frames)}")

00HFP: RGB=747, FLOW=746
00MFE: RGB=494, FLOW=493
00N38: RGB=589, FLOW=588
00NN7: RGB=736, FLOW=735
00SL4: RGB=216, FLOW=215
00T4B: RGB=485, FLOW=484
00X3U: RGB=507, FLOW=506
00YZL: RGB=797, FLOW=796
00ZCA: RGB=2230, FLOW=2229
